In [1]:
# importing relevant libraries
import pandas as pd
import numpy as np
import json
import os
from itertools import chain

# import boto3

from tqdm.notebook import tqdm_notebook
import time

import csv
import re

# display all rows & columns
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# show all outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# Disable warnings
import warnings
warnings.filterwarnings("ignore")

**Import data for analysis**

In [2]:
# Import data
attrition_list = 'Hopeliner Copy of Sample 2022 Attrition List.xlsx'
responses = 'TTEC Interviews (Responses).xlsx'

attrition_df = pd.read_excel(attrition_list)
response_df = pd.read_excel(responses)


**Pre-processing of Dataset**

In [3]:
# Strip column names
attrition_df.columns = [x.strip() for x in attrition_df.columns]
response_df.columns = [x.strip() for x in response_df.columns]

# Response df - pre-processing
response_df.rename(columns =  {'Read Call Opening: \nHi (First Name), thank you for making time for this short interview or chat with us. As shared in our (email/ SMS) exchange, my name is (Interviewer Name). I am from e-BI Solutions. We are working with TTEC to find ways to improve their employee experience so we are grateful that you made time for this.':'Read Call Opening',
                              'Confidentiality Notice and Consent:\nPlease note that we will treat our conversation with confidentiality and only disclose information to the extent you wish to share them with TTEC. Our goal is to help them improve the employee experience through an unbiased study and analysis of the data we gather. We will ask for your sign-off on the final notes we will include in the study. Do you agree and consent to this interview/ short chat? (Share option to record video/ audio/ transcript only)\n\n(Wait for response. Proceed with questions as appropriate)': 'Confidentiality Notice and Consent',
                              }, inplace = True)

cols = ['Stated Reason for Leaving TTEC', 'Other considerations for leaving', 'Top 3 Low lights and impact to them',
'Highlights of their stay with TTEC', 'Suggestions on redesigning the experience',
'What would make you consider returning or referring friends and family members to TTEC?',
'Any questions for e-BI or TTEC?']

for col in cols:
    response_df[col] = response_df[col].astype(str)
    
# Add a new column for all reasons for leaving
response_df['All Leaving Reasons'] = response_df['Stated Reason for Leaving TTEC'] + " " + response_df['Other considerations for leaving']

## Pre-processing Descriptive Data

In [4]:
# pip install gensim
# pip install nltk
# pip install matplotlib
# pip install advertools
# Run 'python -m nltk.downloader all' on ubuntu
# pip install typing-extensions --upgrade

In [5]:
#import modules
import nltk
import gensim
import advertools as adv
import matplotlib.pyplot
import os.path
from gensim import corpora
from gensim.models import LsiModel
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt

# Latent Dirichlet Allocation (LDA) Topic Analysis Model

**Resources:**
- [Evaluate Topic Models: Latent Dirichlet Allocation (LDA)](https://towardsdatascience.com/evaluate-topic-model-in-python-latent-dirichlet-allocation-lda-7d57484bb5d0)

- [pyLDAvis: Topic Modelling Exploration Tool That Every NLP Data Scientist Should Know](https://neptune.ai/blog/pyldavis-topic-modelling-exploration-tool-that-every-nlp-data-scientist-should-know)

In [6]:
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords

# Stop words
stop_words = stopwords.words('english') + list(adv.stopwords['tagalog'])

# Functions
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
        
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

def preprocess_data_indiv(doc_set):
    """
    Input  : docuemnt list
    Purpose: preprocess text (tokenize, removing stopwords, and stemming)
    Output : preprocessed text
    """
    # initialize regex tokenizer
    tokenizer = RegexpTokenizer(r'\w+')
    # create English & Tagalog stop words list
    en_stop = set(stopwords.words('english') + list(adv.stopwords['tagalog']))
    # Create p_stemmer of class PorterStemmer
    p_stemmer = PorterStemmer()
    # list for tokenized documents in loop
    texts = []
    # loop through document list
    for i in doc_set:
        # clean and tokenize document string
        raw = i.lower()
        tokens = tokenizer.tokenize(raw)
        # remove stop words from tokens
        stopped_tokens = [i for i in tokens if not i in en_stop]
        # stem tokens
        #stemmed_tokens = [p_stemmer.stem(i) for i in stopped_tokens]
        # add tokens to list
        texts.append(stopped_tokens)
    return texts


In [7]:
# NLTK Stop words
# import nltk
# nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english') + list(adv.stopwords['tagalog'])
stop_words.extend(['from', 'subject', 're', 'edu', 'use','employee','ttec','resign'])
# Define functions for stopwords, bigrams, trigrams and lemmatization
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]
def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [8]:
# Create a copy of the original response dataframe
analysis_df = response_df.copy()

> 1) Remove punctuations, digits and stop words from text

In [9]:
# Remove punctuations & digits
import re
from string import digits
remove_digits = str.maketrans('', '', digits) # remove digits

for col in cols:
    analysis_df[col] = analysis_df[col].apply(lambda x: re.sub(r'[^\w\s]', '', x).translate(remove_digits).strip().lower())

In [10]:
import spacy
# !python -m spacy download en_core_web_lg 
# !python -m spacy download en_core_web_sm

In [11]:
analysis_df.columns

Index(['Timestamp', 'Email Address', 'Read Call Opening',
       'Confidentiality Notice and Consent', 'Name of former employee',
       'Oracle ID of former employee', 'Last day with TTEC (Complete Date)',
       'Stated Reason for Leaving TTEC', 'Other considerations for leaving',
       'Top 3 Low lights and impact to them',
       'Highlights of their stay with TTEC',
       'Suggestions on redesigning the experience',
       'What would make you consider returning or referring friends and family members to TTEC?',
       'Any questions for e-BI or TTEC?',
       'Closing: Thank the person for the time and trust. Show - responses. End call.',
       'How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good',
       'All Leaving Reasons'],
      dtype='object')

In [12]:
# Remove stop words
data = analysis_df['Suggestions on redesigning the experience'].to_list() # field to analyse (MANUAL INPUT)
data_words = list(sent_to_words(data)) # convert words in response to list of words

# Build the bigram and trigram models
bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
# Faster way to get a sentence clubbed as a trigram/bigram
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)

data_words_nostops = remove_stopwords(data_words) # remove stop words

# Form Bigrams
data_words_bigrams = make_bigrams(data_words_nostops)

# Initialize spacy 'en' model, keeping only tagger component (for efficiency)
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

# Do lemmatization keeping only noun, adj, vb, adv
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

2022-12-21 09:27:48,020 | INFO | phrases.py:497 | learn_vocab | collecting all words and their counts
2022-12-21 09:27:48,021 | INFO | phrases.py:502 | learn_vocab | PROGRESS: at sentence #0, processed 0 words and 0 word types
2022-12-21 09:27:48,024 | INFO | phrases.py:525 | learn_vocab | collected 1693 word types from a corpus of 1437 words (unigram + bigrams) and 107 sentences
2022-12-21 09:27:48,025 | INFO | phrases.py:580 | add_vocab | using 1693 counts as vocab in Phrases<0 vocab, min_count=5, threshold=100, max_vocab_size=40000000>
2022-12-21 09:27:48,026 | INFO | phrases.py:497 | learn_vocab | collecting all words and their counts
2022-12-21 09:27:48,027 | INFO | phrases.py:502 | learn_vocab | PROGRESS: at sentence #0, processed 0 words and 0 word types
2022-12-21 09:27:48,037 | INFO | phrases.py:525 | learn_vocab | collected 1693 word types from a corpus of 1437 words (unigram + bigrams) and 107 sentences
2022-12-21 09:27:48,038 | INFO | phrases.py:580 | add_vocab | using 1693

> 2) Create a corpus

In [13]:
import gensim.corpora as corpora

def prepare_corpus(doc_clean):
    """
    Input  : clean document
    Purpose: create term dictionary of our courpus and Converting list of documents (corpus) into Document Term Matrix
    Output : term dictionary and Document Term Matrix
    """
    # Creating the term dictionary of our courpus, where every unique term is assigned an index. dictionary = corpora.Dictionary(doc_clean)
    dictionary = corpora.Dictionary(doc_clean)
    # Converting list of documents (corpus) into Document Term Matrix using dictionary prepared above.
    doc_term_matrix = [dictionary.doc2bow(doc) for doc in doc_clean]
    # generate LDA model
    return dictionary,doc_term_matrix

In [14]:
import gensim.corpora as corpora
# Create Dictionary
id2word = corpora.Dictionary(data_lemmatized)
# Create Corpus
texts = data_lemmatized
# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in texts]

2022-12-21 09:27:48,970 | INFO | dictionary.py:209 | add_documents | adding document #0 to Dictionary(0 unique tokens: [])
2022-12-21 09:27:48,972 | INFO | dictionary.py:214 | add_documents | built Dictionary(343 unique tokens: ['agent', 'appreciate', 'get', 'good', 'handle']...) from 107 documents (total 705 corpus positions)


> 3) LDA Model 

In [15]:
from pprint import pprint

In [16]:
# number of topics
num_topics = 10

# Build LDA model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                       id2word=id2word,
                                       num_topics=num_topics,
                                      random_state=3,
                                      chunksize=100,
                                       passes=10,
                                       per_word_topics=True)
# Print the Keyword in the 10 topics
pprint(lda_model.print_topics())
doc_lda = lda_model[corpus]

2022-12-21 09:27:48,982 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric alpha at 0.1
2022-12-21 09:27:48,983 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric eta at 0.1
2022-12-21 09:27:48,984 | INFO | ldamodel.py:481 | __init__ | using serial LDA version on this node
2022-12-21 09:27:48,986 | INFO | ldamulticore.py:238 | update | running online LDA training, 10 topics, 10 passes over the supplied corpus of 107 documents, updating every 700 documents, evaluating every ~107 documents, iterating 50x with a convergence threshold of 0.001000
2022-12-21 09:27:48,995 | INFO | ldamulticore.py:279 | update | training LDA model using 7 processes
2022-12-21 09:27:49,057 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:49,063 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:51,15

2022-12-21 09:27:51,237 | INFO | ldamodel.py:1171 | show_topics | topic #4 (0.100): 0.139*"none" + 0.030*"employee" + 0.028*"agent" + 0.017*"especially" + 0.017*"training" + 0.017*"proper" + 0.017*"mag" + 0.017*"hire" + 0.012*"need" + 0.012*"work"
2022-12-21 09:27:51,238 | INFO | ldamodel.py:1171 | show_topics | topic #5 (0.100): 0.025*"agent" + 0.025*"system" + 0.025*"high" + 0.025*"tag" + 0.013*"employee" + 0.013*"job" + 0.013*"inform" + 0.013*"change" + 0.013*"well" + 0.013*"work"
2022-12-21 09:27:51,238 | INFO | ldamodel.py:1171 | show_topics | topic #9 (0.100): 0.053*"employee" + 0.034*"good" + 0.020*"pay" + 0.020*"time" + 0.020*"even" + 0.014*"incentive" + 0.014*"concern" + 0.014*"management" + 0.014*"ask" + 0.014*"company"
2022-12-21 09:27:51,239 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.100): 0.081*"salary" + 0.043*"employee" + 0.036*"basic" + 0.031*"offer" + 0.029*"increase" + 0.023*"training" + 0.023*"agent" + 0.022*"competitive" + 0.016*"wfh" + 0.016*"back"
2022-

2022-12-21 09:27:51,314 | INFO | ldamodel.py:1171 | show_topics | topic #1 (0.100): 0.034*"process" + 0.034*"agent" + 0.034*"training" + 0.034*"considerate" + 0.018*"pay" + 0.018*"additional" + 0.018*"leader" + 0.018*"work" + 0.018*"engagement" + 0.018*"team"
2022-12-21 09:27:51,315 | INFO | ldamodel.py:1049 | do_mstep | topic diff=0.005766, rho=0.315127
2022-12-21 09:27:51,317 | INFO | ldamodel.py:822 | log_perplexity | -6.726 per-word bound, 105.8 perplexity estimate based on a held-out corpus of 7 documents with 33 words
2022-12-21 09:27:51,318 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:51,319 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:51,330 | INFO | ldamodel.py:1171 | show_topics | topic #7 (0.100): 0.042*"people" + 0.022*"agent" + 0.022*"ensure" + 0.022*"hire" + 0.022*"skil

[(0,
  '0.082*"salary" + 0.043*"employee" + 0.038*"basic" + 0.030*"offer" + '
  '0.030*"increase" + 0.023*"training" + 0.023*"agent" + 0.023*"competitive" + '
  '0.016*"wfh" + 0.016*"back"'),
 (1,
  '0.034*"process" + 0.034*"agent" + 0.034*"training" + 0.034*"considerate" + '
  '0.018*"pay" + 0.018*"leader" + 0.018*"additional" + 0.018*"work" + '
  '0.018*"engagement" + 0.018*"team"'),
 (2,
  '0.028*"health" + 0.028*"employee" + 0.028*"care" + 0.028*"cover" + '
  '0.015*"stress" + 0.015*"take" + 0.015*"help" + 0.015*"area" + 0.015*"night" '
  '+ 0.015*"focus"'),
 (3,
  '0.045*"employee" + 0.031*"emergency" + 0.016*"change" + 0.016*"basic" + '
  '0.016*"take" + 0.016*"competitive" + 0.016*"base" + 0.016*"concern" + '
  '0.016*"soon" + 0.016*"schedule"'),
 (4,
  '0.144*"none" + 0.030*"employee" + 0.028*"agent" + 0.017*"especially" + '
  '0.017*"training" + 0.017*"proper" + 0.017*"mag" + 0.017*"hire" + '
  '0.012*"need" + 0.012*"work"'),
 (5,
  '0.025*"agent" + 0.025*"system" + 0.025*"hig

> 4) Select optimal number of topics

In [17]:
# supporting function
def compute_coherence_values(corpus, dictionary, k, a, b):
    
    lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                           id2word=dictionary,
                                           num_topics=k, 
                                           random_state=100,
                                           chunksize=100,
                                           passes=10,
                                           alpha=a,
                                           eta=b)
    
    coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
    
    return coherence_model_lda.get_coherence()

In [18]:
# Baseline coherence score
from gensim.models import CoherenceModel
# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nBaseline Coherence Score: ', coherence_lda)

2022-12-21 09:27:51,390 | INFO | probability_estimation.py:155 | p_boolean_sliding_window | using ParallelWordOccurrenceAccumulator(processes=7, batch_size=64) to estimate probabilities from sliding windows
2022-12-21 09:27:53,214 | INFO | text_analysis.py:530 | terminate_workers | 7 accumulators retrieved from output queue
2022-12-21 09:27:53,247 | INFO | text_analysis.py:552 | merge_accumulators | accumulated word occurrence stats for 97 virtual documents



Baseline Coherence Score:  0.3697795102210873


In [19]:
# Optimize coherence score
import tqdm
grid = {}
grid['Validation_Set'] = {}
# Topics range
min_topics = 2
max_topics = 11
step_size = 1
topics_range = range(min_topics, max_topics, step_size)

# Alpha parameter
alpha = list(np.arange(0.01, 1, 0.3))
alpha.append('symmetric')
alpha.append('asymmetric')

# Beta parameter
beta = list(np.arange(0.01, 1, 0.3))
beta.append('symmetric')

# Validation sets
num_of_docs = len(corpus)
corpus_sets = [
#     gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.25)), 
#                gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.5)), 
               gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.75)), 
               corpus]

corpus_title = [
    #'25% Corpus','50% Corpus',
    '75% Corpus', '100% Corpus']

model_results = {'Validation_Set': [],
                 'Topics': [],
                 'Alpha': [],
                 'Beta': [],
                 'Coherence': []
                }

In [20]:
# # iterate through validation corpuses
# for i in tqdm_notebook(range(len(corpus_sets))):
#     # iterate through number of topics
#     for k in topics_range:
#         # iterate through alpha values
#         for a in alpha:
#             # iterare through beta values
#             for b in beta:
#                 # get the coherence score for the given parameters
#                 cv = compute_coherence_values(corpus=corpus_sets[i], dictionary=id2word, 
#                                               k=k, a=a, b=b)
#                 # Save the model results
#                 model_results['Validation_Set'].append(corpus_title[i])
#                 model_results['Topics'].append(k)
#                 model_results['Alpha'].append(a)
#                 model_results['Beta'].append(b)
#                 model_results['Coherence'].append(cv)


In [21]:
# # Create coherence dataframe
# coherence_results = (pd.DataFrame(model_results)
#                      .sort_values(by = 'Coherence')
#                     )

In [22]:
# # Plot results
# plot_df = (coherence_results
#              #.query("(Alpha != 'asymmetric') & (Alpha != 'symmetric') ", engine = 'python')
#              #.query("(Validation_Set != '25% Corpus') & (Validation_Set != '50% Corpus')", engine = 'python')
#              .query("(Alpha == 'symmetric') & (Beta == 'symmetric')", engine = 'python')
#              .sort_values(by = 'Topics', ascending = False)             
#             )
# plot_df

# # Plot
# plt.plot(plot_df['Topics'].to_list(), plot_df['Coherence'].to_list())
# plt.show()

In [23]:
# # Select optimal alpha & beta
# (coherence_results
#  .query("Topics == 3")
#  .sort_values(by = 'Coherence', ascending = False)
# )

> 5) Train final model with optimized hyperparameters

In [24]:
# Train final model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=3, 
                                           random_state=3,
                                           chunksize=100,
                                           passes=10,
                                           alpha='symmetric',
                                           eta=0.01)

2022-12-21 09:27:53,639 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric alpha at 0.3333333333333333
2022-12-21 09:27:53,640 | INFO | ldamodel.py:481 | __init__ | using serial LDA version on this node
2022-12-21 09:27:53,641 | INFO | ldamulticore.py:238 | update | running online LDA training, 3 topics, 10 passes over the supplied corpus of 107 documents, updating every 700 documents, evaluating every ~107 documents, iterating 50x with a convergence threshold of 0.001000
2022-12-21 09:27:53,642 | INFO | ldamulticore.py:279 | update | training LDA model using 7 processes
2022-12-21 09:27:53,696 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:53,700 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:55,719 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.333): 0.055*"employee" + 

2022-12-21 09:27:55,858 | INFO | ldamodel.py:1049 | do_mstep | topic diff=3.300632, rho=0.376089
2022-12-21 09:27:55,860 | INFO | ldamodel.py:822 | log_perplexity | -7.979 per-word bound, 252.2 perplexity estimate based on a held-out corpus of 7 documents with 33 words
2022-12-21 09:27:55,861 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 6, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:55,863 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 6, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:55,880 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.333): 0.053*"employee" + 0.036*"salary" + 0.024*"increase" + 0.022*"basic" + 0.021*"agent" + 0.016*"good" + 0.016*"pay" + 0.016*"training" + 0.015*"wfh" + 0.014*"incentive"
2022-12-21 09:27:55,881 | INFO | ldamodel.py:1171 | show_topics | topic #1 (0.333): 0.054*"employee" + 0.033*"work" + 0.027*"especially" + 0.024*"hire" + 0.021*"train" +

> 4) Visualizing topics

For (1), you can manually select each topic to view its top most frequent and/or “relevant” terms, using different values of the λ parameter. This can help when you’re trying to assign a human interpretable name or “meaning” to each topic.

For (2), exploring the Intertopic Distance Plot can help you learn about how topics relate to each other, including potential higher-level structure between groups of topics.

**Interpretation:**


*Bar chart*
- Each bubble represents a topic. The larger the bubble, the higher percentage of the number of responses in the corpus is about that topic.
- Blue bars represent the overall frequency of each word in the corpus. If no topic is selected, the blue bars of the most frequently used words will be displayed.
- Red bars give the estimated number of times a given term was generated by a given topic. The word with the longest red bar is the word that is used the most by the responses belonging to that topic.
- <u>*Lambda*</u> -- Decreasing the lambda parameter, increases the weight of the ratio of the frequency of word given the topic / Overall frequency of the word in the documents. Important words for the given topic moves upward.
    - Lambda = 1, top words by frequency
    - Lambda = 0, top words by relevance to topic

*Bubble plot*
- The further the bubbles are away from each other, the more different they are. 
- The larger the bubble, the more frequent is the topic in the documents.
- Distance between the topics is an approximation of semantic relationship between the topics.
- The topic which shares common words will be overlapping (closer in distance) in comparison to the non-overlapping topic.
- A good topic model will have big and non-overlapping bubbles scattered throughout the chart. As we can see from the graph, the bubbles are clustered within one place.
- A topic model with a low number of topics will have big non-overlapping bubbles, scattered throughout the chart whereas, the topic model with a high number of topics, will have many overlapping small size bubbles, clustered in the chart.


In [25]:
import pyLDAvis.gensim
import pickle 
import pyLDAvis

In [26]:
# Visualize the topics
pyLDAvis.enable_notebook()
LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
LDAvis_prepared

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.060231 -0.239822       1        1  51.270341
1     -0.222407  0.162812       2        1  24.916423
2      0.282638  0.077010       3        1  23.813236, topic_info=          Term       Freq      Total Category  logprob  loglift
33        none  25.000000  25.000000  Default  30.0000  30.0000
0        agent  15.000000  15.000000  Default  29.0000  29.0000
156     salary  13.000000  13.000000  Default  28.0000  28.0000
24        work   7.000000   7.000000  Default  27.0000  27.0000
155   increase   8.000000   8.000000  Default  26.0000  26.0000
..         ...        ...        ...      ...      ...      ...
115  incentive   1.740809   6.787940   Topic3  -4.5689   0.0741
24        work   1.525260   7.257267   Topic3  -4.7011  -0.1249
113       well   1.098433   5.999955   Topic3  -5.0294  -0.2629
224   possible   1.008541   1.961727   Topic3  -5.1148   0.7696
50      inform   1.022277   3.012487   Topic3  -5.1012   0.3542

[186 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
68        1  0.770870     account
68        3  0.256957     account
229       3  1.018374    activity
218       1  1.020549  additional
25        2  0.644494      adjust
...     ...       ...         ...
113       2  0.333336        well
113       3  0.166668        well
40        1  1.031054         wfh
24        2  0.826758        work
24        3  0.275586        work

[186 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3])

# Post-modeling Exploratory Data Analysis

*Assign respondents to their topic clusters*

In [27]:
# Topic distribution for the whole document. Each element in the list is a pair of a topic’s id, and the probability that was assigned to it.
topic_dist = list(lda_model.get_document_topics(corpus))


# Classify responses to topics
cluster = []
probability = []

for i in range(0, len(topic_dist)):
    clust = [x for x,y in topic_dist[i]]
    prob = [y for x,y in topic_dist[i]]
    get_index = prob.index(max(prob))
    max_val = topic_dist[i][get_index]
    cluster.append(max_val[0])
    probability.append(max_val[1])
    
# Add classification to df
analysis_df['Topic'] = cluster
analysis_df['Topic_Probability'] = probability

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [28]:
# Map topics to Oracle ID
id_topic_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['Topic']))
id_prob_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['Topic_Probability']))
id_rate_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']))

# For those with missing Oracle ID, map to names
name_mapper = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','Topic']]
                
                     .set_index("Name").to_dict()['Topic']
)

name_mapper2 = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','Topic_Probability']]
                
                     .set_index("Name").to_dict()['Topic_Probability']
)

rating_mapper = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']]
                
                     .set_index("Name").to_dict()['How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']
)

# Filter attrition df to contain respondents only
respondents_demog_df = (attrition_df
                         .assign(Topic1 = lambda x: x['EE Oracle Id'].map(id_topic_mapper),
                                 Prob1 = lambda x: x['EE Oracle Id'].map(id_prob_mapper),
                                 Rate1 = lambda x: x['EE Oracle Id'].map(id_rate_mapper),
                                 Name = lambda x: (x['First Name'] + " " + x['Last Name']).apply(lambda x: x.upper()),
                                 Topic = lambda x: np.where(x['Topic1'].isna(), x['Name'].map(name_mapper), x['Topic1']),
                                 Topic_Probability = lambda x: np.where(x['Prob1'].isna(), x['Name'].map(name_mapper2), x['Prob1']),
                                Rating = lambda x: np.where(x['Rate1'].isna(), x['Name'].map(rating_mapper), x['Rate1']),

                         )
                         .drop(['Topic1','Prob1','Rate1'], axis = 1)
                         .query("Topic.notna()", engine = 'python')
                         .sort_values(by = 'Topic')
                        )

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


> 1) How many respondents belong to each topic cluster & what is the average probability score?

In [29]:
# "Number of Employees per Cluster & Average Probability Score")
gen_stats = (analysis_df
             .groupby(['Topic']).agg({'Oracle ID of former employee':'nunique', 'Topic_Probability':'describe'})
             .reset_index()
            )

gen_stats

print("Number of Employees per Cluster & Average Probability Score")


/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Topic Oracle ID of former employee Topic_Probability                      \
        Oracle ID of former employee             count      mean       std   
0     0                           51              51.0  0.781300  0.216304   
1     1                           17              17.0  0.872360  0.118505   
2     2                           39              39.0  0.747992  0.116962   

                                                     
        min       25%       50%       75%       max  
0  0.333333  0.643994  0.888859  0.931351  0.974862  
1  0.540734  0.860864  0.900423  0.944389  0.979257  
2  0.571086  0.666666  0.666666  0.858302  0.959044

Number of Employees per Cluster & Average Probability Score


> 2) Tenure, Topic Probability, and Employee Count by By Attrition Stage

In [30]:
(respondents_demog_df
 .groupby(['Topic','Stage Attrition']).agg({'EE Oracle Id':'nunique','Tenure In Months':['mean', 'std','median'],'Topic_Probability':['mean', 'std','median']})
 .unstack('Stage Attrition')
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


EE Oracle Id            Tenure In Months             \
                       nunique                        mean              
Stage Attrition Pre Production Production   Pre Production Production   
Topic                                                                   
0.0                          6         45         2.166667  25.244444   
1.0                          3         14         2.333333  32.142857   
2.0                          5         34        11.000000  23.441176   

                                                                     \
                           std                    median              
Stage Attrition Pre Production Production Pre Production Production   
Topic                                                                 
0.0                   1.602082  19.151496            1.5       21.0   
1.0                   1.527525  32.646963            2.0       24.0   
2.0                  13.619838  18.818416            2.0       18.0   

                Topic_Probability                                       \
                             mean                       std              
Stage Attrition    Pre Production Production Pre Production Production   
Topic                                                                    
0.0                      0.695314   0.792764       0.285808   0.206742   
1.0                      0.814662   0.884724       0.138375   0.115783   
2.0                      0.780691   0.743183       0.156220   0.112289   

                                           
                        median             
Stage Attrition Pre Production Production  
Topic                                      
0.0                   0.821298   0.901145  
1.0                   0.860864   0.910093  
2.0                   0.666666   0.666666

> 3) Tenure, Topic Probability, and Employee Count by By Job Name

In [31]:
(respondents_demog_df
 .groupby(['Topic','Job Name','Stage Attrition']).agg({'EE Oracle Id':'nunique','Tenure In Months':['mean', 'std','median']})
 .unstack('Job Name')
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


EE Oracle Id                                        \
                           nunique                                         
Job Name                     CSR I CSR II CSR III Chat Sales Associate I   
Topic Stage Attrition                                                      
0.0   Pre Production           3.0    3.0     NaN                    NaN   
      Production               7.0   28.0     7.0                    NaN   
1.0   Pre Production           2.0    1.0     NaN                    NaN   
      Production               1.0    9.0     3.0                    1.0   
2.0   Pre Production           1.0    4.0     NaN                    NaN   
      Production               5.0   24.0     3.0                    NaN   

                                          Tenure In Months             \
                                                      mean              
Job Name              ISR I TSR I TSR III            CSR I     CSR II   
Topic Stage Attrition                                                   
0.0   Pre Production    NaN   NaN     NaN         3.333333   1.000000   
      Production        1.0   1.0     1.0        30.142857  22.392857   
1.0   Pre Production    NaN   NaN     NaN         3.000000   1.000000   
      Production        NaN   NaN     NaN         2.000000  35.333333   
2.0   Pre Production    NaN   NaN     NaN         1.000000  13.500000   
      Production        NaN   NaN     2.0        38.600000  20.708333   

                                                                             \
                                                                              
Job Name                 CSR III Chat Sales Associate I ISR I TSR I TSR III   
Topic Stage Attrition                                                         
0.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       22.142857                    NaN  59.0  56.0    28.0   
1.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       30.000000                   40.0   NaN   NaN     NaN   
2.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       25.666667                    NaN   NaN   NaN    15.0   

                                                                               \
                             std                                                
Job Name                   CSR I     CSR II    CSR III Chat Sales Associate I   
Topic Stage Attrition                                                           
0.0   Pre Production    1.527525   0.000000        NaN                    NaN   
      Production       17.033580  19.343559  17.023793                    NaN   
1.0   Pre Production    1.414214        NaN        NaN                    NaN   
      Production             NaN  39.550601  11.532563                    NaN   
2.0   Pre Production         NaN  14.341083        NaN                    NaN   
      Production       18.609138  19.268084  11.590226                    NaN   

                                                                   \
                                            median                  
Job Name              ISR I TSR I   TSR III  CSR I CSR II CSR III   
Topic Stage Attrition                                               
0.0   Pre Production    NaN   NaN       NaN    3.0    1.0     NaN   
      Production        NaN   NaN       NaN   24.0   20.0    16.0   
1.0   Pre Production    NaN   NaN       NaN    3.0    1.0     NaN   
      Production        NaN   NaN       NaN    2.0   22.0    26.0   
2.0   Pre Production    NaN   NaN       NaN    1.0   11.5     NaN   
      Production        NaN   NaN  2.828427   43.0   16.0    20.0   

                                                                  
                                                                  
Job Name              Chat Sales Associate I ISR I TSR I TSR III  
Topic Stage Attrition               

> 4) Tenure, Topic Probability, and Employee Count by By Job Name

In [32]:
(respondents_demog_df
 .groupby(['Topic','Stage Attrition']).agg({'Rating': ['mean', 'std','median']})
 .round(1)
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Rating            
                        mean  std median
Topic Stage Attrition                   
0.0   Pre Production     4.0  1.0    4.0
      Production         4.0  0.8    4.0
1.0   Pre Production     3.5  0.7    3.5
      Production         4.1  1.0    4.0
2.0   Pre Production     4.8  0.5    5.0
      Production         4.5  0.6    4.0

In [33]:
respondents_demog_df['Tenure In Months'].describe()

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


count    107.000000
mean      22.971963
std       21.379084
min        1.000000
25%        6.500000
50%       19.000000
75%       29.500000
max      127.000000
Name: Tenure In Months, dtype: float64